# Final inference — XGBoost v4 champion

Loads the champion **XGBoost v4** model from DagsHub/MLflow (run `train_and_blend`) and writes a Kaggle `submission.csv`.

Unlike PatchTST-style `model.predict(test)`, XGBoost needs:
1. Clean + static FE on train/test
2. Lag state from **full train history** (`SeriesStore`)
3. **Recursive** week-by-week forecast (predictions feed next lags)
4. Blend with `lag_52` (`α = 0.75`) + Christmas mass-shift (`1/7`)

**Kaggle:** attach competition data · **Internet On** · CPU · Save & Run All  
If the MLflow model cannot be loaded, the notebook **retrains** with the logged champion hyperparameters.

In [ ]:
#1 — install + imports
!pip install -q dagshub mlflow xgboost

import os
import time
from collections import deque, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

import dagshub
import mlflow
import mlflow.xgboost

pd.set_option("display.max_columns", 50)
print("xgboost", xgb.__version__)

In [ ]:
#2 — paths + champion constants (from experiment-xgboost-v4 Optuna / blend search)
CANDIDATE_DIRS = [
    Path("/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"),
    Path("/kaggle/input/walmart-recruiting-store-sales-forecasting"),
    Path("."),
]
DATA_DIR = next((p for p in CANDIDATE_DIRS if (p / "train.csv").exists() or (p / "train.csv.zip").exists()), CANDIDATE_DIRS[0])
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
CHRISTMAS_SHIFT_FRACTION = 1 / 7
BEST_ALPHA = 0.75  # lag_52 blend from v4 val search

# DagsHub run: XGBoost_v4_Training / train_and_blend
MODEL_URI = "runs:/fce07372a2ed45c097f0547236c1e642/model"
RETRAIN_IF_LOAD_FAILS = True

CHAMPION_PARAMS = dict(
    objective="reg:absoluteerror",
    n_estimators=400,
    learning_rate=0.113387,
    max_depth=9,
    min_child_weight=30,
    subsample=0.77307,
    colsample_bytree=0.75562,
    enable_categorical=True,
    tree_method="hist",
    random_state=RANDOM_STATE,
)

# Feature set kept by v4 importance > 0 (Type, Size, IsChristmas dropped as model inputs)
SELECTED_FEATURES = [
    "lag_1", "lag_2", "lag_52", "rolling_mean_4", "rolling_mean_8", "dept_woy_avg_sales",
    "WeekOfYear", "lag_4", "IsHoliday", "Dept", "IsThanksgiving", "Month", "storedept_avg_sales",
    "MarkDown3", "Year", "rolling_std_4", "Store", "MarkDown4", "IsSuperBowl", "Fuel_Price",
    "MarkDown2", "MarkDown1", "lag_8", "MarkDown5", "Unemployment", "CPI", "Temperature", "IsLaborDay",
]
SELECTED_CAT_FEATURES = [
    c for c in ["Store", "Dept", "IsHoliday", "IsThanksgiving", "IsSuperBowl", "IsLaborDay"]
    if c in SELECTED_FEATURES
]

print("DATA_DIR:", DATA_DIR)
print("MODEL_URI:", MODEL_URI)
print("n_features:", len(SELECTED_FEATURES), "| alpha:", BEST_ALPHA)

In [ ]:
#3 — DagsHub / MLflow
dagshub.init(repo_owner="lshek22", repo_name="walmart-recruiting-store-sales-forecasting", mlflow=True)
mlflow.set_experiment("XGBoost_v4_Inference")

model = None
model_source = None
try:
    model = mlflow.xgboost.load_model(MODEL_URI)
    model_source = f"mlflow:{MODEL_URI}"
    print("Loaded champion XGBoost from MLflow:", MODEL_URI)
except Exception as e:
    print(f"MLflow load failed: {type(e).__name__}: {e}")
    if not RETRAIN_IF_LOAD_FAILS:
        raise
    print("Will retrain with CHAMPION_PARAMS after feature engineering.")

In [ ]:
#4 — load competition data
def _read_csv(name, **kwargs):
    zipped = DATA_DIR / f"{name}.csv.zip"
    plain = DATA_DIR / f"{name}.csv"
    path = zipped if zipped.exists() else plain
    return pd.read_csv(path, **kwargs)

train_raw = _read_csv("train", parse_dates=["Date"])
test_raw = _read_csv("test", parse_dates=["Date"])
stores = _read_csv("stores")
features = _read_csv("features", parse_dates=["Date"])
features["IsHoliday"] = features["IsHoliday"].astype(bool)

print("train:", train_raw.shape, "test:", test_raw.shape, "stores:", stores.shape, "features:", features.shape)
train_raw.head()

In [ ]:
#5 — cleaning + feature engineering (same as experiment-xgboost-v4)
HOLIDAY_DATES = {
    "SuperBowl": ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
    "LaborDay": ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
    "Thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
    "Christmas": ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
}
HOLIDAY_FLAG_COLS = [f"Is{name}" for name in HOLIDAY_DATES]
LAG_COLS = [
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_52",
    "rolling_mean_4", "rolling_mean_8", "rolling_std_4",
    "storedept_avg_sales", "dept_woy_avg_sales",
]
CATEGORICAL_FEATURES = ["Store", "Dept", "Type"] + HOLIDAY_FLAG_COLS + ["IsHoliday"]


def clean_features(features_df):
    df = features_df.copy()
    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)
    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].transform(lambda s: s.ffill().bfill())
    return df


def add_static_features(df, stores_df, features_df):
    out = df.merge(stores_df, on="Store", how="left")
    out = out.merge(features_df.drop(columns=["IsHoliday"]), on=["Store", "Date"], how="left")
    for name, dates in HOLIDAY_DATES.items():
        out[f"Is{name}"] = out["Date"].isin(pd.to_datetime(dates))
    out["Year"] = out["Date"].dt.year
    out["Month"] = out["Date"].dt.month
    out["WeekOfYear"] = out["Date"].dt.isocalendar().week.astype(int)
    out["IsHoliday"] = out["IsHoliday"].astype(bool)
    # keep a real bool for Christmas post-process (category cast would break .astype(bool))
    out["IsChristmas_bool"] = out["IsChristmas"].astype(bool)
    return out


def add_bulk_lag_features(df):
    df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
    g = df.groupby(["Store", "Dept"])["Weekly_Sales"]
    df["lag_1"] = g.shift(1)
    df["lag_2"] = g.shift(2)
    df["lag_4"] = g.shift(4)
    df["lag_8"] = g.shift(8)
    df["lag_52"] = g.shift(52)
    df["rolling_mean_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).mean()
    )
    df["rolling_mean_8"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.shift(1).rolling(8, min_periods=1).mean()
    )
    df["rolling_std_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).std()
    )
    df["storedept_avg_sales"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.expanding().mean().shift(1)
    )
    tmp = df.sort_values(["Dept", "WeekOfYear", "Year"])
    tmp["dept_woy_avg_sales"] = tmp.groupby(["Dept", "WeekOfYear"])["Weekly_Sales"].transform(
        lambda s: s.expanding().mean().shift(1)
    )
    return tmp.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)


features_clean = clean_features(features)
train_static = add_static_features(train_raw, stores, features_clean)
test_static = add_static_features(test_raw, stores, features_clean)
train_m = add_bulk_lag_features(train_static)

fillna_medians = {c: float(train_m[c].median()) for c in LAG_COLS}
for c in LAG_COLS:
    train_m[c] = train_m.groupby(["Store", "Dept"])[c].transform(lambda s: s.fillna(s.median()))
    train_m[c] = train_m[c].fillna(fillna_medians[c])

for c in CATEGORICAL_FEATURES:
    if train_m[c].dtype == "bool":
        train_m[c] = train_m[c].astype(str)
        test_static[c] = test_static[c].astype(str)
    train_m[c] = train_m[c].astype("category")
    test_static[c] = test_static[c].astype("category")

missing = [c for c in SELECTED_FEATURES if c not in train_m.columns]
assert not missing, f"Missing features on train: {missing}"
print(f"train_m={train_m.shape} | test_static={test_static.shape} | fillna_medians ok")

In [ ]:
#6 — SeriesStore + recursive forecast (same as experiment-xgboost-v4)
class SeriesStore:
    def __init__(self):
        self.recent = defaultdict(lambda: deque(maxlen=8))
        self.by_date = {}
        self.sd_sum_count = defaultdict(lambda: [0.0, 0])
        self.woy_sum_count = defaultdict(lambda: [0.0, 0])

    def get_features(self, store, dept, date, woy):
        key = (store, dept)
        recent = self.recent[key]
        n = len(recent)
        lag_1 = recent[-1] if n >= 1 else np.nan
        lag_2 = recent[-2] if n >= 2 else np.nan
        lag_4 = recent[-4] if n >= 4 else np.nan
        lag_8 = recent[-8] if n >= 8 else np.nan
        lag_52 = self.by_date.get((store, dept, date - pd.Timedelta(weeks=52)), np.nan)
        rolling_mean_4 = float(np.mean(list(recent)[-4:])) if n >= 1 else np.nan
        rolling_mean_8 = float(np.mean(list(recent))) if n >= 1 else np.nan
        rolling_std_4 = float(np.std(list(recent)[-4:], ddof=0)) if n >= 2 else np.nan
        s, c = self.sd_sum_count[key]
        storedept_avg = s / c if c > 0 else np.nan
        ws, wc = self.woy_sum_count[(dept, woy)]
        dept_woy_avg = ws / wc if wc > 0 else np.nan
        return dict(
            lag_1=lag_1, lag_2=lag_2, lag_4=lag_4, lag_8=lag_8, lag_52=lag_52,
            rolling_mean_4=rolling_mean_4, rolling_mean_8=rolling_mean_8,
            rolling_std_4=rolling_std_4, storedept_avg_sales=storedept_avg,
            dept_woy_avg_sales=dept_woy_avg,
        )

    def add(self, store, dept, date, woy, value):
        key = (store, dept)
        self.recent[key].append(value)
        self.by_date[(store, dept, date)] = value
        self.sd_sum_count[key][0] += value
        self.sd_sum_count[key][1] += 1
        self.woy_sum_count[(dept, woy)][0] += value
        self.woy_sum_count[(dept, woy)][1] += 1


def build_series_store(history_df):
    store = SeriesStore()
    for row in history_df.itertuples(index=False):
        store.add(row.Store, row.Dept, row.Date, row.WeekOfYear, row.Weekly_Sales)
    return store


def recursive_forecast(model, series_store, future_df, feature_cols, categorical_features, fillna_medians):
    preds_all = []
    for dt, day_rows in future_df.groupby("Date", sort=True):
        feat_dicts = [
            series_store.get_features(r.Store, r.Dept, dt, r.WeekOfYear)
            for r in day_rows.itertuples(index=False)
        ]
        feat_df = pd.DataFrame(feat_dicts, index=day_rows.index)
        merged = pd.concat([day_rows, feat_df], axis=1)

        for c in LAG_COLS:
            merged[c] = merged[c].fillna(fillna_medians.get(c, 0.0))
        for c in categorical_features:
            if merged[c].dtype == bool:
                merged[c] = merged[c].astype(str)
            merged[c] = merged[c].astype("category")

        preds = model.predict(merged[feature_cols])
        merged["Weekly_Sales_pred"] = preds
        preds_all.append(merged)

        for r, p in zip(day_rows.itertuples(index=False), preds):
            series_store.add(r.Store, r.Dept, dt, r.WeekOfYear, float(p))

    return pd.concat(preds_all, ignore_index=True)


def christmas_shift_correction(df, preds, shift_fraction=CHRISTMAS_SHIFT_FRACTION):
    preds = np.asarray(preds, dtype=float).copy()
    df = df.reset_index(drop=True)
    if "IsChristmas_bool" in df.columns:
        xmas = df["IsChristmas_bool"].astype(bool).values
    else:
        xmas = df["IsChristmas"].astype(str).str.lower().isin(["true", "1"]).values
    for i in np.where(xmas)[0]:
        store, dept, date = df.loc[i, "Store"], df.loc[i, "Dept"], df.loc[i, "Date"]
        prev_mask = (df["Store"] == store) & (df["Dept"] == dept) & (df["Date"] == date - pd.Timedelta(weeks=1))
        prev_idx = df.index[prev_mask]
        if len(prev_idx) == 1:
            shift_amt = preds[i] * shift_fraction
            preds[i] -= shift_amt
            preds[prev_idx[0]] += shift_amt
    return preds

print("Helpers ready")

In [ ]:
#7 — ensure model (load already tried; else retrain champion)
if model is None:
    print("Retraining final XGBoost with CHAMPION_PARAMS…")
    if train_m["IsHoliday"].dtype.name == "category":
        hol = train_m["IsHoliday"].astype(str).str.lower().isin(["true", "1"])
    else:
        hol = train_m["IsHoliday"].astype(bool)
    w_full = np.where(hol, 5, 1)
    model = xgb.XGBRegressor(**CHAMPION_PARAMS)
    model.fit(train_m[SELECTED_FEATURES], train_m["Weekly_Sales"], sample_weight=w_full)
    model_source = "retrain:CHAMPION_PARAMS"
    print("Retrain done.")
else:
    print("Using loaded model:", model_source)

In [ ]:
#8 — recursive test forecast → blend → Christmas shift → submission.csv
with mlflow.start_run(run_name="xgboost_v4_inference"):
    mlflow.log_params({
        "model_source": model_source,
        "model_uri": MODEL_URI,
        "best_alpha": BEST_ALPHA,
        "christmas_shift_fraction": CHRISTMAS_SHIFT_FRACTION,
        "n_selected_features": len(SELECTED_FEATURES),
        "notebook": "model-inference-final",
    })

    t0 = time.time()
    series_store_test = build_series_store(
        train_m[["Store", "Dept", "Date", "WeekOfYear", "Weekly_Sales"]]
    )
    test_preds_df = recursive_forecast(
        model,
        series_store_test,
        test_static,
        SELECTED_FEATURES,
        SELECTED_CAT_FEATURES,
        fillna_medians,
    )
    forecast_s = time.time() - t0
    print(f"Recursive forecast done in {forecast_s:.1f}s | rows={len(test_preds_df)}")

    if "lag_52" in test_preds_df.columns:
        test_blend = (
            BEST_ALPHA * test_preds_df["Weekly_Sales_pred"].values
            + (1 - BEST_ALPHA) * test_preds_df["lag_52"].values
        )
    else:
        test_blend = test_preds_df["Weekly_Sales_pred"].values

    test_preds_reset = test_preds_df.reset_index(drop=True)
    test_final_preds = christmas_shift_correction(
        test_preds_reset, test_blend, CHRISTMAS_SHIFT_FRACTION
    )

    submission = test_preds_reset.copy()
    submission["Weekly_Sales"] = test_final_preds
    submission["Id"] = (
        submission["Store"].astype(int).astype(str) + "_"
        + submission["Dept"].astype(int).astype(str) + "_"
        + submission["Date"].dt.strftime("%Y-%m-%d")
    )
    submission = submission[["Id", "Weekly_Sales"]]

    submission_path = OUTPUT_DIR / "submission_xgb_v4_inference.csv"
    submission.to_csv(submission_path, index=False)
    # also write root submission.csv for convenient Kaggle download
    submission.to_csv("submission.csv", index=False)

    mlflow.log_metric("test_forecast_seconds", forecast_s)
    mlflow.log_metric("n_submission_rows", len(submission))
    mlflow.log_artifact(str(submission_path))

print(f"Saved {len(submission)} rows → {submission_path} and ./submission.csv")
submission.head()